In [2]:
!pip install git+https://github.com/florencejt/fusilli.git
!pip install nibabel

  Cloning https://github.com/florencejt/fusilli.git to c:\users\bnish\appdata\local\temp\pip-req-build-n6ki5dav
  Resolved https://github.com/florencejt/fusilli.git to commit bbd29f94f9ec43c22d8e15e87a05b86fad25752f
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/florencejt/fusilli.git 'C:\Users\bnish\AppData\Local\Temp\pip-req-build-n6ki5dav'
  Running command git submodule update --init --recursive -q


In [3]:
import os
import sys
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch import tensor
import numpy as np
import nibabel as nib
import glob

from fusilli.data import prepare_fusion_data
from fusilli.train import train_and_save_models
from fusilli.fusionmodels.tabularimagefusion.concat_img_maps_tabular_maps import ConcatImageMapsTabularMaps
from fusilli.eval import RealsVsPreds, ConfusionMatrix

In [4]:
## Data preprocessing
print('cwd (base):', os.getcwd(),'\n')

# open and load csv
base_dir = Path(r"C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0\ATLAS_2")  
meta_xlsx  = base_dir / "20220425_ATLAS_2.0_MetaData.xlsx"          
train_dir  = base_dir / "Training"  
test_dir   = base_dir / "Testing" 

meta_df = pd.read_excel(meta_xlsx)

cwd (base): C:\Users\bnish 



In [5]:
def norm_sub_id(x):
    s = str(x).strip()
    return s.replace("sub-","")

meta_df["ID"] = meta_df["Subject ID"].apply(norm_sub_id)

In [6]:
# T1w image paths
def find_t1_for_subject(sub_id: str) -> str | None:
    patterns = [
        str(train_dir / f"**/*{sub_id}*/*anat/*T1w*.nii*"),
        str(train_dir / f"**/*{sub_id}*/*T1w*.nii*"),
        str(train_dir / f"**/*{sub_id}*T1w*.nii*"),
    ]
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return None

meta_df["image_path"] = meta_df["ID"].apply(find_t1_for_subject)

before = len(meta_df)
meta_df = meta_df.dropna(subset=["image_path"]).reset_index(drop=True)
after = len(meta_df)
print(f"Fixed T1w paths for {after}/{before} subjects in Training")

Fixed T1w paths for 655/655 subjects in Training


In [7]:
#Labels
import re

def normalize_loc_string(s: str):
    s = str(s).strip().lower()
    if not s or s in {"nan", "none"}:
        return []
    s = re.sub(r"[;/]", ",", s)
    s = re.sub(r"[^a-z, ]", "", s)
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return sorted(set(parts))

if "Primary Stroke Location" not in meta_df.columns:
    raise ValueError("MetaData.xlsx is missing 'Primary Stroke Location'.")

meta_df["loc_tokens"] = meta_df["Primary Stroke Location"].apply(normalize_loc_string)

def choose_primary(tokens):
    if not tokens:
        return np.nan
    return tokens[0]

meta_df["primary_loc"] = meta_df["loc_tokens"].apply(choose_primary)

loc_vocab = sorted(meta_df["primary_loc"].dropna().unique())
loc2id = {loc: i for i, loc in enumerate(loc_vocab)}
print("Location classes (label -> region):")
for loc, idx in loc2id.items():
    print(f"  {idx}: {loc}")

meta_df["prediction_label"] = meta_df["primary_loc"].map(loc2id)

before = len(meta_df)
meta_df = meta_df.dropna(subset=["prediction_label"]).reset_index(drop=True)
after = len(meta_df)
print(f"Kept {after}/{before} subjects with a valid primary location label.")

Location classes (label -> region):
  0: basal ganglia
  1: brainstem
  2: cerebellum
  3: frontal lobe
  4: hippocampus
  5: insula
  6: occipital lobe
  7: parietal lobe
  8: temporal lobe
  9: thalamus
Kept 655/655 subjects with a valid primary location label.


In [8]:
# New csv for mapped data for later
mapped_csv = base_dir / "mapped_atlas_metadata.csv"
meta_df.to_csv(mapped_csv, index=False)
print(f"Saved mapped metadata in {mapped_csv}...\n")
print(meta_df[["ID","image_path","Primary Stroke Hemisphere" if "Primary Stroke Hemisphere" in meta_df.columns else "Lesion Volume"]].head())

Saved mapped metadata in C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0\ATLAS_2\mapped_atlas_metadata.csv...

         ID                                         image_path  \
0  r001s001  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   
1  r001s002  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   
2  r001s003  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   
3  r001s004  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   
4  r001s005  C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0...   

  Primary Stroke Hemisphere  
0                     Right  
1                     Right  
2                     Right  
3                      Left  
4                      Left  


In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])     



def grab_slice(nifti_path: Path, grid=(6,6)) -> Image.Image:
    ni = nib.load(str(nifti_path))
    vol = ni.get_fdata()
    vol = np.nan_to_num(vol, nan=0.0)

    H, W, Z = vol.shape
    r, c = grid
    n = r * c

    finite = vol[np.isfinite(vol)]
    if finite.size == 0:
        lo, hi = 0.0, 1.0
    else:
        lo, hi = np.percentile(finite, [1, 99])
        if hi <= lo:
            hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo + 1e-8), 0, 1)

    if Z >= n:
        z_idxs = np.linspace(0, Z - 1, n, dtype=int)
    else:
        # If fewer slices than needed, repeat some
        z_idxs = np.round(np.linspace(0, Z - 1, n)).astype(int)

    tiles = []
    for z in z_idxs:
        sl = (vol[:, :, z] * 255.0).astype(np.uint8) 
        tiles.append(sl)

    tile_h, tile_w = H, W 

    montage = np.zeros((r * tile_h, c * tile_w), dtype=np.uint8)
    k = 0
    for i in range(r):
        for j in range(c):
            sl = tiles[k]
            # place tile
            y0, y1 = i * tile_h, (i + 1) * tile_h
            x0, x1 = j * tile_w, (j + 1) * tile_w
            montage[y0:y1, x0:x1] = sl
            k += 1

    return Image.fromarray(montage) 

df = pd.read_csv(mapped_csv)
images, valid_idx = [], []
GRID = (6, 6)

for i, row in tqdm(df.iterrows(), total=len(df)):
    p = Path(row["image_path"])
    if not p.exists():
        continue

    pil_img = grab_slice(p, grid=GRID)
    images.append(transform(pil_img))
    valid_idx.append(i)

image_tensor = torch.stack(images)
torch.save(image_tensor, base_dir / "images.pt")

df = df.loc[valid_idx].reset_index(drop=True)
df.to_csv(mapped_csv, index=False)

print("Saved image tensor:", base_dir / "images.pt", "shape:", image_tensor.shape)


100%|██████████████████████████████████████████████████████████████████████████████| 655/655 [2:44:43<00:00, 15.09s/it]


Saved image tensor: C:\Users\bnish\Downloads\ATLAS_R2.0\ATLAS_R2.0\ATLAS_2\images.pt shape: torch.Size([655, 1, 224, 224])


In [10]:
df_numeric = pd.read_csv(mapped_csv)
non_feature_cols = {"ID", "prediction_label"}

def to_numeric_series(s: pd.Series) -> pd.Series:
    if s.dtype.kind in "biufc":
        return s
    str_s = s.astype(str).str.strip()
    first_number = str_s.str.extract(r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)", expand=False)
    out = pd.to_numeric(first_number, errors="coerce")
    return out

clean = {}
for c in df_numeric.columns:
    if c in non_feature_cols:
        clean[c] = df_numeric[c]
    else:
        clean[c] = to_numeric_series(df_numeric[c])

clean = pd.DataFrame(clean)

feature_cols = [c for c in clean.columns if c not in non_feature_cols]
min_non_na = max(10, int(0.6 * len(clean))) 
kept = []
dropped = []
for c in feature_cols:
    if clean[c].notna().sum() >= min_non_na:
        kept.append(c)
    else:
        dropped.append(c)
clean = clean[["ID", "prediction_label"] + kept].dropna().reset_index(drop=True)

clean[kept] = clean[kept].astype("float32")
clean["prediction_label"] = clean["prediction_label"].astype("int64")

tabular_csv = base_dir / "mapped_atlas_numeric.csv"
clean.to_csv(tabular_csv, index=False)

print(f"[Tabular cleanup] kept {len(kept)} numeric feature columns; dropped {len(dropped)}: {dropped}")
print(f"[Tabular cleanup] final shape: {clean.shape}")



data_paths = {
    "tabular1": str(tabular_csv),
    "tabular2": "",
    "image": str(base_dir / "images.pt"),
}

[Tabular cleanup] kept 13 numeric feature columns; dropped 13: ['Organism', 'Organism Part', 'Developmental Stage', 'Primary Stroke Hemisphere', 'Primary Stroke Location', 'Secondary Stroke Hemisphere', 'Secondary Stroke Location', 'Scanner Brand', 'ATLAS 1.2 Subject ID', 'INDI Subject ID', 'Chronicity', 'loc_tokens', 'primary_loc']
[Tabular cleanup] final shape: (544, 15)


In [11]:
output_paths = {
    "checkpoints": "outputs/checkpoints",
    "losses": "outputs/losses",
    "figures": "outputs/figures",
}
for p in output_paths.values():
    os.makedirs(p, exist_ok=True)

In [12]:
counts = clean["prediction_label"].value_counts().sort_index()
inv_freq = (counts.sum() / (len(counts) * counts)).values
class_weights = torch.tensor(inv_freq, dtype=torch.float32)
print("class_weights:", class_weights.tolist())

num_classes = clean["prediction_label"].nunique()
print("Number of location classes:", num_classes)

data_module = prepare_fusion_data(
    prediction_task="multiclass",
    fusion_model=ConcatImageMapsTabularMaps,
    data_paths=data_paths,
    output_paths=output_paths,
    batch_size=8,
    test_size=0.2,
    multiclass_dimensions=num_classes,
    image_downsample_size=(224, 224),
    num_workers=4,
)


class_weights: [0.19638989865779877, 1.3948718309402466, 1.9428571462631226, 0.4000000059604645, 18.133333206176758, 2.8631579875946045, 2.0923078060150146, 4.9454545974731445, 13.600000381469727, 54.400001525878906]
Number of location classes: 10


In [ ]:
trained_model = train_and_save_models(
    data_module=data_module,
    fusion_model=ConcatImageMapsTabularMaps,
    training_modifications={
        "accelerator": "cpu",
        "devices": 1,       # number of GPUs
         "loss_params": {
        #     "pos_weight": pos_weight
        "weight": class_weights },
        # "precision": 16,    # mixed precision
    },
    max_epochs=100,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
C:\Users\bnish\anaconda3\Lib\site-packages\lightning\fabric\loggers\csv_logs.py:268: Experiment logs directory outputs/losses\ConcatImageMapsTabularMaps exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!
C:\Users\bnish\anaconda3\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:751: Checkpoint directory C:\Users\bnish\outputs\checkpoints exists and is not empty.

  | Name  | Type                       | Params | Mode 
-------------------------------------------------------------
0 | model | ConcatImageMapsTabularMaps | 2.8 M  | train
-------------------------------------------------------------
2.8 M     Trainable params
0         Non-trainable params
2.8 M     Total params
11.236    Total estimated model params size (MB)
47        Modules in train mode
0         Modules in eval mode
C:\Users\bnish\anacond

Training: |                                                                                      | 0/? [00:00<…

C:\Users\bnish\anaconda3\Lib\site-packages\torchmetrics\utilities\prints.py:36: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)
C:\Users\bnish\anaconda3\Lib\site-packages\torchmetrics\utilities\prints.py:36: UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)


In [ ]:
RealsVsPreds.from_final_val_data(trained_model)
ConfusionMatrix.from_final_val_data(trained_model)
plt.show()

In [ ]:
# trained_model is what train_and_save_models returned.
# Often it's a list; if so, grab the first element.
model = trained_model[0] if isinstance(trained_model, list) else trained_model

# Look at what the model exposes
print([a for a in dir(model) if "val" in a.lower()])
